# CDN on MovieLens-1M - Cross Decoupling Network


In [ ]:
CFG = {
    "fast_run": False,
    "final_epochs": 40,
    "cdn_tune_epochs": 20,
    "temperature": 1.0,
    "seeds": [0],
    "batch_size": 1024,
    "embed_dim": 64,
    "tower_hidden": [256, 128, 64],
    "expert_hidden": 64,
    "grad_clip": 1.0,
    "max_title_words": 2000,
    "head_fraction": 0.2,
    "eval_k": [10, 50],
    "eval_user_batch_size": 512,
    "eval_item_batch_size": 1024,
}

# Fixed from our current best MovieLens Two-Tower baseline
LR, DROPOUT, WD = 0.01, 0.0, 0.0

# Paper B.3: gamma in {1.2, 1.5, 2.1, 3, 5, 10, 30}; expert count in {1, 2, 3}
GAMMA_GRID = [1.2, 1.5, 2.1, 3.0, 5.0, 10.0, 30.0]
EXPERT_COUNT_GRID = [1, 2, 3]

if CFG["fast_run"]:
    CFG.update({
        "final_epochs": 2,
        "cdn_tune_epochs": 2,
        "seeds": [0],
        "batch_size": 512,
        "max_title_words": 500,
    })
    GAMMA_GRID = [1.5]
    EXPERT_COUNT_GRID = [1]
    print("FAST_RUN")

print(f"CDN backbone: lr={LR}, dropout={DROPOUT}, wd={WD}")
print(f"Tune: {CFG['cdn_tune_epochs']} epochs; final: {CFG['final_epochs']} epochs")
print("gamma grid:", GAMMA_GRID)
print("expert count grid:", EXPERT_COUNT_GRID)


In [ ]:
# !pip install -q pandas numpy requests tqdm

In [ ]:
import copy, itertools, json, math, re, zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

def set_seed(s):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
def find_or_download_ml1m(work_dir="/kaggle/working") -> Path:
    kin = Path("/kaggle/input")
    if kin.exists():
        for p in kin.rglob("ratings.dat"):
            root = p.parent
            if (root / "movies.dat").exists() and (root / "users.dat").exists():
                print("Kaggle input:", root); return root
    for p in [Path("ml-1m"), Path(work_dir) / "ml-1m", Path("data/ml-1m")]:
        if (p / "ratings.dat").exists():
            print("Local:", p); return p
    out = Path(work_dir)
    url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
    print("Downloading..."); zpath = out / "ml-1m.zip"
    r = requests.get(url, timeout=180); r.raise_for_status()
    zpath.write_bytes(r.content)
    with zipfile.ZipFile(zpath) as zf: zf.extractall(out)
    root = out / "ml-1m"; print("Saved:", root); return root

RAW_DIR = find_or_download_ml1m()

In [ ]:
GENRES = [
    "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical",
    "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western",
]

def read_ml1m(raw: Path):
    users = pd.read_csv(raw / "users.dat", sep="::", engine="python",
        names=["user_id", "gender", "age", "occupation", "zip"], encoding="latin-1")
    ratings = pd.read_csv(raw / "ratings.dat", sep="::", engine="python",
        names=["user_id", "item_id", "rating", "timestamp"], encoding="latin-1")
    ratings = ratings[ratings["rating"] > 0]
    movies = []
    
    with open(raw / "movies.dat", encoding="latin-1") as f:
        for line in f:
            parts = line.strip().split("::")
            mid, title = int(parts[0]), parts[1]
            genres = parts[2].split("|") if len(parts) > 2 else []
            year = 1995
            
            if title.endswith(")") and "(" in title:
                try:
                    year = int(title[title.rfind("(") + 1 : -1])
                    
                except ValueError:
                    pass
            
            movies.append({"item_id": mid, "title": title, "genres": genres, "year": year})
            
    return users, pd.DataFrame(movies), ratings

users_df, movies_df, ratings = read_ml1m(RAW_DIR)
item_ids = sorted(ratings["item_id"].unique())
uid_map = {u: i for i, u in enumerate(users_df["user_id"].tolist())}
iid_map = {m: i for i, m in enumerate(item_ids)}
NUM_USERS, NUM_ITEMS = len(uid_map), len(iid_map)
print(f"users={NUM_USERS} items={NUM_ITEMS} ratings={len(ratings)}")

In [ ]:
def tokenize_title(title):
    if "(" in title: title = title.rsplit("(", 1)[0]
    return re.findall(r"[a-z0-9]+", title.lower())

def build_title_bow(titles, max_words):
    counter = Counter()
    per = []
    
    for t in titles:
        toks = tokenize_title(t)
        per.append(toks)
        counter.update(toks)
        
    vocab = [w for w, _ in counter.most_common(max_words)]
    w2i = {w: i for i, w in enumerate(vocab)}
    mat = np.zeros((len(titles), len(vocab)), np.float32)
    for r, toks in enumerate(per):
        for tok in toks:
            j = w2i.get(tok)
            if j is not None:
                mat[r, j] = 1.0
            
    return mat

g2i = {g: i for i, g in enumerate(GENRES)}
gm, years, titles = [], [], []
for mid in item_ids:
    row = movies_df.loc[movies_df["item_id"] == mid].iloc[0]
    v = np.zeros(len(GENRES), np.float32)
    
    for g in row["genres"]:
        if g in g2i:
            v[g2i[g]] = 1.0
        
    gm.append(v)
    years.append(row["year"])
    titles.append(row["title"])
    
years = np.array(years, np.float32)
years = (years - years.mean()) / (years.std() + 1e-6)
bow = build_title_bow(titles, CFG["max_title_words"])
ITEM_GEN = np.hstack([np.stack(gm), years.reshape(-1, 1), bow]).astype(np.float32)
mu, sig = ITEM_GEN.mean(0, keepdims=True), ITEM_GEN.std(0, keepdims=True) + 1e-6
ITEM_GEN = ((ITEM_GEN - mu) / sig).astype(np.float32)
ITEM_GEN_DIM = ITEM_GEN.shape[1]

gm_u = {g: i for i, g in enumerate(sorted(users_df["gender"].unique()))}
am_u = {a: i for i, a in enumerate(sorted(users_df["age"].unique()))}
om_u = {o: i for i, o in enumerate(sorted(users_df["occupation"].unique()))}
zm_u = {z: i for i, z in enumerate(sorted(users_df["zip"].unique()))}
USER_GEN = np.stack([
    users_df["gender"].map(gm_u), users_df["age"].map(am_u),
    users_df["occupation"].map(om_u), users_df["zip"].map(zm_u),
], axis=1).astype(np.int64)
USER_GEN_SIZES = [len(gm_u), len(am_u), len(om_u), len(zm_u)]
print(f"item_gen dim={ITEM_GEN_DIM}")

In [ ]:
train_pairs, val_u, val_i, test_u, test_i = [], [], [], [], []
train_user_items = [set() for _ in range(NUM_USERS)]

for uid, grp in ratings.groupby("user_id"):
    if uid not in uid_map:
        continue
        
    u = uid_map[uid]
    items = [iid_map[i] for i in grp.sort_values("timestamp")["item_id"] if i in iid_map]
    
    if len(items) < 3:
        continue
        
    for it in items[:-2]:
        train_pairs.append((u, it))
        train_user_items[u].add(it)
        
    val_u.append(u); val_i.append(items[-2])
    test_u.append(u); test_i.append(items[-1])

train_pairs = np.array(train_pairs, np.int64)
val_u, val_i = np.array(val_u, np.int64), np.array(val_i, np.int64)
test_u, test_i = np.array(test_u, np.int64), np.array(test_i, np.int64)
print(f"train={len(train_pairs)} val={len(val_u)} test={len(test_u)}")

known_user_items_val = [set(s) for s in train_user_items]

known_user_items_test = [set(s) for s in train_user_items]
for u, i in zip(val_u, val_i):
    known_user_items_test[int(u)].add(int(i))


item_freq = np.zeros(NUM_ITEMS, np.int64)
for _, i in train_pairs: item_freq[i] += 1
order = np.argsort(-item_freq)
n_head = max(1, int(NUM_ITEMS * CFG["head_fraction"]))
head_mask = np.zeros(NUM_ITEMS, bool); head_mask[order[:n_head]] = True
print(f"head={head_mask.sum()} tail={(~head_mask).sum()}")

In [ ]:
def make_regularizer_pairs(seed=0):
    rng = np.random.default_rng(seed)
    pairs_by_item = defaultdict(list)
    for u, i in train_pairs:
        pairs_by_item[int(i)].append((int(u), int(i)))

    tail_freq = item_freq[~head_mask]
    tail_freq = tail_freq[tail_freq > 0]
    max_tail_freq = int(tail_freq.max()) if len(tail_freq) else 1

    reg_pairs = []
    for i in range(NUM_ITEMS):
        pairs = pairs_by_item.get(i, [])
        if not pairs:
            continue
        if head_mask[i] and len(pairs) > max_tail_freq:
            idx = rng.choice(len(pairs), size=max_tail_freq, replace=False)
            reg_pairs.extend([pairs[j] for j in idx])
        else:
            reg_pairs.extend(pairs)

    rng.shuffle(reg_pairs)
    return np.array(reg_pairs, dtype=np.int64), max_tail_freq


reg_pairs, max_tail_freq = make_regularizer_pairs(seed=0)
print(f"Omega_m train pairs={len(train_pairs)}")
print(f"Omega_r regularizer pairs={len(reg_pairs)} max_tail_freq={max_tail_freq}")
print(f"Omega_r ratio={len(reg_pairs) / max(len(train_pairs), 1):.3f}")


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, dropout=0.0):
        super().__init__()
        d = in_dim
        layers = []
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d = h
        self.net = nn.Sequential(*layers)
        self.out_dim = d

    def forward(self, x):
        return self.net(x)


class UserGenEnc(nn.Module):
    def __init__(self, sizes, dim=16):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(n, dim) for n in sizes])
        self.out_dim = len(sizes) * dim

    def forward(self, x):
        return torch.cat([e(x[:, i].long()) for i, e in enumerate(self.embs)], -1)


class CDN(nn.Module):
    def __init__(self, dropout=0.0, n_mem_experts=1, n_gen_experts=1):
        super().__init__()
        ed = CFG["embed_dim"]
        h = CFG["tower_hidden"]
        expert_hidden = CFG["expert_hidden"]
        self.n_mem_experts = n_mem_experts
        self.n_gen_experts = n_gen_experts
        self.out_dim = h[-1]

        self.user_id = nn.Embedding(NUM_USERS, ed)
        self.user_gen = UserGenEnc(USER_GEN_SIZES, 16)
        user_in = ed + self.user_gen.out_dim
        self.user_shared = MLP(user_in, [h[0]], dropout)
        self.user_main = MLP(h[0], h[1:], dropout)
        self.user_reg = MLP(h[0], h[1:], dropout)
        self.user_main_ln = nn.LayerNorm(self.out_dim)
        self.user_reg_ln = nn.LayerNorm(self.out_dim)

        self.item_id = nn.Embedding(NUM_ITEMS, ed)
        self.mem_experts = nn.ModuleList([MLP(ed, [expert_hidden], dropout) for _ in range(n_mem_experts)])
        self.gen_experts = nn.ModuleList([MLP(ITEM_GEN_DIM, [expert_hidden], dropout) for _ in range(n_gen_experts)])
        self.mem_proj = nn.Linear(expert_hidden, self.out_dim)
        self.gen_proj = nn.Linear(expert_hidden, self.out_dim)
        self.gate = nn.Linear(1, n_mem_experts + n_gen_experts)
        self.item_ln = nn.LayerNorm(self.out_dim)

    def user_base(self, u, ug):
        x = torch.cat([self.user_id(u), self.user_gen(ug)], -1)
        return self.user_shared(x)

    def user_vec_main(self, u, ug):
        return self.user_main_ln(self.user_main(self.user_base(u, ug)))

    def user_vec_reg(self, u, ug):
        return self.user_reg_ln(self.user_reg(self.user_base(u, ug)))

    def item_vec_moe(self, i, ig):
        id_x = self.item_id(i)
        mem = torch.stack([self.mem_proj(expert(id_x)) for expert in self.mem_experts], dim=1)
        gen = torch.stack([self.gen_proj(expert(ig)) for expert in self.gen_experts], dim=1)
        experts = torch.cat([mem, gen], dim=1)
        weights = torch.softmax(self.gate(item_pop_t[i].unsqueeze(-1)), dim=-1).unsqueeze(-1)
        y = (experts * weights).sum(dim=1)
        return self.item_ln(y)

    def user_vec(self, u, ug):
        return self.user_vec_main(u, ug)

    def item_vec(self, i, ig):
        return self.item_vec_moe(i, ig)

    def cdn_logits(self, main_users, main_items, reg_users, reg_items, alpha):
        xm = F.normalize(self.user_vec_main(main_users, ug_t[main_users]), dim=-1, eps=1e-6)
        ym = F.normalize(self.item_vec_moe(main_items, ig_t[main_items]), dim=-1, eps=1e-6)
        xr = F.normalize(self.user_vec_reg(reg_users, ug_t[reg_users]), dim=-1, eps=1e-6)
        yr = F.normalize(self.item_vec_moe(reg_items, ig_t[reg_items]), dim=-1, eps=1e-6)
        logits_m = xm @ ym.T
        logits_r = xr @ yr.T
        logits = alpha * logits_m + (1.0 - alpha) * logits_r
        logits = logits / CFG.get("temperature", 1.0)
        logits = torch.clamp(logits, -20.0, 20.0)
        if not torch.isfinite(logits).all():
            raise RuntimeError("NaN/Inf in CDN logits")
        return logits


class PairDS(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u, i = self.pairs[idx]
        return int(u), int(i)


ug_t = torch.from_numpy(USER_GEN).long().to(device)
ig_t = torch.from_numpy(ITEM_GEN).float().to(device)
item_pop = np.log1p(item_freq.astype(np.float32))
item_pop = (item_pop - item_pop.mean()) / (item_pop.std() + 1e-6)
item_pop_t = torch.from_numpy(item_pop.astype(np.float32)).to(device)

main_loader = DataLoader(PairDS(train_pairs), batch_size=CFG["batch_size"], shuffle=True, drop_last=True)
reg_loader = DataLoader(PairDS(reg_pairs), batch_size=CFG["batch_size"], shuffle=True, drop_last=True)
print("CDN helpers ready")


In [ ]:
@torch.no_grad()
def evaluate(model, users, items, ks, head_mask, known_user_items):
    model.eval()
    eval_user_batch_size = CFG.get("eval_user_batch_size", 512)
    eval_item_batch_size = CFG.get("eval_item_batch_size", 1024)

    item_vecs_all = []
    for s in tqdm(range(0, NUM_ITEMS, eval_item_batch_size), desc="item vecs", leave=False):
        e = min(s + eval_item_batch_size, NUM_ITEMS)
        idx = torch.arange(s, e, device=device)
        item_vecs_all.append(F.normalize(model.item_vec(idx, ig_t[idx]), dim=-1, eps=1e-6))
    item_vecs_all = torch.cat(item_vecs_all, dim=0)

    ranks_all, ranks_h, ranks_t = [], [], []
    users_np = np.asarray(users, dtype=np.int64)
    items_np = np.asarray(items, dtype=np.int64)

    for start in tqdm(range(0, len(users_np), eval_user_batch_size), desc="eval"):
        end = min(start + eval_user_batch_size, len(users_np))
        batch_users_np = users_np[start:end]
        batch_true_np = items_np[start:end]
        batch_users = torch.from_numpy(batch_users_np).long().to(device)
        batch_true = torch.from_numpy(batch_true_np).long().to(device)

        user_vecs = F.normalize(model.user_vec(batch_users, ug_t[batch_users]), dim=-1, eps=1e-6)
        scores = user_vecs @ item_vecs_all.T

        for row, (u, true_i) in enumerate(zip(batch_users_np, batch_true_np)):
            blocked = [it for it in known_user_items[int(u)] if int(it) != int(true_i)]
            if blocked:
                scores[row, torch.tensor(blocked, device=device, dtype=torch.long)] = -1e9

        true_scores = scores[torch.arange(scores.size(0), device=device), batch_true]
        ranks = (scores > true_scores.unsqueeze(1)).sum(dim=1).detach().cpu().numpy().astype(np.int64)

        for rank, true_i in zip(ranks, batch_true_np):
            ranks_all.append(int(rank))
            (ranks_h if head_mask[int(true_i)] else ranks_t).append(int(rank))

    del item_vecs_all
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    def agg(ranks):
        a = np.array(ranks, dtype=np.int64)
        out = {}
        for k in ks:
            hits = a < k
            out[f"HR@{k}"] = 100 * hits.mean() if len(a) else np.nan
            out[f"NDCG@{k}"] = 100 * np.where(hits, 1 / np.log2(a + 2), 0).mean() if len(a) else np.nan
        return out

    return {"overall": agg(ranks_all), "head": agg(ranks_h), "tail": agg(ranks_t)}


def metric_line(metrics, prefix=""):
    parts = []
    for split in ("overall", "head", "tail"):
        parts.append(
            f"{split}: HR@50={metrics[split]['HR@50']:.2f}, "
            f"NDCG@50={metrics[split]['NDCG@50']:.2f}"
        )
    print(prefix + " | ".join(parts))


def summarize_runs(all_metrics, title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    for split in ("overall", "head", "tail"):
        for k in CFG["eval_k"]:
            for mk in [f"HR@{k}", f"NDCG@{k}"]:
                vals = [r[split][mk] for r in all_metrics]
                vals = [v for v in vals if np.isfinite(v)]
                if len(vals) == 0:
                    print(f"  [{split}] {mk}: nan")
                    continue
                mean = np.mean(vals)
                se = np.std(vals, ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0
                print(f"  [{split}] {mk}: {mean:.2f} ± {se:.2f}")


def alpha_for_epoch(epoch, total_epochs, gamma):
    return 1.0 - (epoch / (gamma * total_epochs)) ** 2


def train_cdn(model, opt, epochs, gamma, desc="cdn"):
    for ep in range(1, epochs + 1):
        model.train()
        alpha = alpha_for_epoch(ep, epochs, gamma)
        total, n = 0.0, 0
        reg_iter = iter(reg_loader)
        pbar = tqdm(main_loader, desc=f"{desc} {ep}/{epochs} alpha={alpha:.3f}", leave=False)

        for main_users, main_items in pbar:
            try:
                reg_users, reg_items = next(reg_iter)
            except StopIteration:
                reg_iter = iter(reg_loader)
                reg_users, reg_items = next(reg_iter)

            main_users = main_users.to(device)
            main_items = main_items.to(device)
            reg_users = reg_users.to(device)
            reg_items = reg_items.to(device)

            logits = model.cdn_logits(main_users, main_items, reg_users, reg_items, alpha)
            labels = torch.arange(logits.size(0), device=device)
            loss = F.cross_entropy(logits, labels)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at epoch={ep}")

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            opt.step()

            total += loss.item() * main_users.size(0)
            n += main_users.size(0)
            pbar.set_postfix(loss=f"{loss.item():.3f}")

        print(f"  {desc} ep{ep} alpha={alpha:.3f} loss={total/max(n,1):.4f}")


def run_cdn(gamma, n_experts, epochs, seed=0, split="val"):
    set_seed(seed)
    model = CDN(dropout=DROPOUT, n_mem_experts=n_experts, n_gen_experts=n_experts).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    train_cdn(model, opt, epochs, gamma, desc=f"cdn gamma={gamma} experts={n_experts}")

    if split == "val":
        metrics = evaluate(model, val_u, val_i, CFG["eval_k"], head_mask, known_user_items_val)
    elif split == "test":
        metrics = evaluate(model, test_u, test_i, CFG["eval_k"], head_mask, known_user_items_test)
    else:
        raise ValueError(split)

    metric_line(metrics, prefix=f"cdn gamma={gamma} experts={n_experts} {split}: ")
    return metrics


In [ ]:
cdn_results = []
print("\n" + "=" * 60)
print("Grid search: CDN gamma + expert count")
print("=" * 60)

for n_experts in EXPERT_COUNT_GRID:
    for gamma in GAMMA_GRID:
        metrics = run_cdn(gamma, n_experts, CFG["cdn_tune_epochs"], seed=CFG["seeds"][0], split="val")
        score = metrics["overall"]["HR@50"]
        cdn_results.append({
            "gamma": gamma,
            "n_experts": n_experts,
            "val_HR@50": score,
            "metrics": metrics,
        })
        print(f"candidate gamma={gamma} experts={n_experts}: val overall HR@50={score:.2f}")

best_cdn = max(cdn_results, key=lambda x: x["val_HR@50"])
print("\nBest CDN:", {k: best_cdn[k] for k in ["gamma", "n_experts", "val_HR@50"]})


In [ ]:
GAMMA = best_cdn["gamma"]
N_EXPERTS = best_cdn["n_experts"]
print(f"Final CDN: gamma={GAMMA} n_experts={N_EXPERTS} lr={LR} dropout={DROPOUT} wd={WD}")

all_test = []
for seed in CFG["seeds"]:
    print(f"\n{'='*50}\nSeed {seed}\n{'='*50}")
    metrics = run_cdn(GAMMA, N_EXPERTS, CFG["final_epochs"], seed=seed, split="test")
    all_test.append(metrics)
    for split in ("overall", "head", "tail"):
        print(split, metrics[split])

summarize_runs(all_test, f"TEST mean ± stderr: CDN gamma={GAMMA} experts={N_EXPERTS}")
